# Reusable Process Graphs and the Web Editor

Every openEO analysis is a **process graph**: a JSON document describing which processes to run and how they connect. You can save a process graph on the server as a **user-defined process (UDP)** with named parameters, then call it again with different areas or dates, from Python or from the [FAO OpenEO Web Editor](https://data.fao.org/editor/?server=data.fao.org%2Fopeneo).

**What you will learn:**
- How to view a process graph as JSON
- How to declare parameters and save a UDP with `save_user_defined_process`
- How to list, inspect and call your saved processes
- How to combine a UDP with other processes
- How to load a process graph from JSON (for example, one exported from the Web Editor)
- How to delete a UDP

**Prerequisites:**
- Completed the [Getting Started](01_getting_started.ipynb) tutorial
- `openeo` and `pandas` packages installed

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/un-fao/openeo-docs/blob/main/tutorials/08_reusable_process_graphs.ipynb)

In [1]:
# !pip install openeo pandas

## 1. Connect and Authenticate

UDPs are stored per user, so you must be signed in. Only you can see and call your own UDPs.

In [2]:
import json
import openeo
import pandas as pd
from openeo.api.process import Parameter

connection = openeo.connect("https://data.fao.org/openeo")
connection.authenticate_oidc(provider_id="keycloak")
print("Authenticated.")

Authenticated using refresh token.
Authenticated.


## 2. A Process Graph Is JSON

Here is a small analysis: total CHIRPS rainfall over a fixed area and period. `to_json()` shows the process graph the Python client sends to the server.

In [3]:
CHIRPS = "fao-gismgr:CHIRPS-V3:raster:mapsets:EWX2-DATA-GM"

fixed = connection.load_collection(
    CHIRPS,
    spatial_extent={"west": 36.5, "east": 37.5, "south": -1.5, "north": -0.5},
    temporal_extent=["2022-10-01", "2023-01-01"],
).reduce_dimension(dimension="time", reducer="sum")

print(fixed.to_json())

{
  "process_graph": {
    "loadcollection1": {
      "process_id": "load_collection",
      "arguments": {
        "id": "fao-gismgr:CHIRPS-V3:raster:mapsets:EWX2-DATA-GM",
        "spatial_extent": {
          "west": 36.5,
          "east": 37.5,
          "south": -1.5,
          "north": -0.5
        },
        "temporal_extent": [
          "2022-10-01",
          "2023-01-01"
        ]
      }
    },
    "reducedimension1": {
      "process_id": "reduce_dimension",
      "arguments": {
        "data": {
          "from_node": "loadcollection1"
        },
        "dimension": "time",
        "reducer": {
          "process_graph": {
            "sum1": {
              "process_id": "sum",
              "arguments": {
                "data": {
                  "from_parameter": "data"
                }
              },
              "result": true
            }
          }
        }
      },
      "result": true
    }
  }
}


This JSON is exactly what the Web Editor shows in its **Process Graph** (code) view. You can paste it there, or copy a graph from the Editor and run it from Python (see section 6).

The area and dates are hard-coded. To reuse the analysis, we replace them with **parameters**.

## 3. Declare Parameters and Save a UDP

`Parameter` objects act as placeholders in the graph. Helper constructors such as `Parameter.spatial_extent`, `Parameter.temporal_interval` and `Parameter.geojson` add the right JSON schema, so the Web Editor can show suitable input widgets (a map for areas, date pickers for intervals).

In [4]:
spatial_extent = Parameter.spatial_extent(name="spatial_extent", description="Bounding box to analyse")
temporal_extent = Parameter.temporal_interval(name="temporal_extent", description="Period [start, end); end is excluded")
geometries = Parameter.geojson(name="geometries", description="Polygons to summarise")

rain_total = connection.load_collection(
    CHIRPS,
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
).reduce_dimension(dimension="time", reducer="sum")

rain_by_zone = rain_total.process(
    "zonal_stats",
    data_cube=rain_total,
    geometries=geometries,
    stats=["mean", "max"],
)

Save it with an ID made of letters, digits and underscores. Saving again with the same ID overwrites the existing UDP.

In [5]:
udp = connection.save_user_defined_process(
    user_defined_process_id="rainfall_total_by_zone",
    process_graph=rain_by_zone,
    parameters=[spatial_extent, temporal_extent, geometries],
    summary="Total CHIRPS v3 rainfall per polygon",
    description="Sums monthly CHIRPS v3 precipitation over the period and returns mean and max per polygon.",
)
print(f"Saved UDP: {udp.user_defined_process_id}")

Saved UDP: rainfall_total_by_zone


## 4. List and Inspect Your UDPs

In [6]:
for p in connection.list_user_defined_processes():
    print(f"{p['id']:30s} {p.get('summary', '')}")

rainfall_total_by_zone         Total CHIRPS v3 rainfall per polygon


In [7]:
details = udp.describe()
for param in details["parameters"]:
    print(f"- {param['name']}: {param['description']}")
print(f"\nProcess graph nodes: {[n['process_id'] for n in details['process_graph'].values()]}")

- spatial_extent: Bounding box to analyse
- temporal_extent: Period [start, end); end is excluded
- geometries: Polygons to summarise

Process graph nodes: ['load_collection', 'reduce_dimension', 'zonal_stats']


## 5. Call the UDP with Different Inputs

`datacube_from_process` calls any process by ID, including your UDPs. Pass a value for **every** parameter: the FAO backend doesn't fill in parameter defaults, so a missing parameter causes a `ProcessGraphError`.

We reuse the Kenya county boundaries from the [Drought Monitoring](07_drought_monitoring.ipynb) tutorial and compare the October–December short rains of 2022 (a drought year) and 2023 (an El Niño year).

In [8]:
import os
import urllib.request

COUNTIES_PATH = "data/kenya_counties.geojson"
if not os.path.exists(COUNTIES_PATH):
    os.makedirs("data", exist_ok=True)
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/un-fao/openeo-docs/main/tutorials/data/kenya_counties.geojson",
        COUNTIES_PATH,
    )
with open(COUNTIES_PATH) as f:
    counties = json.load(f)

selected = {"type": "FeatureCollection",
            "features": [f for f in counties["features"]
                         if f["properties"]["county"] in ("Turkana", "Wajir", "Kitui", "Nakuru")]}
kenya_bbox = {"west": 33.9, "east": 41.95, "south": -4.7, "north": 5.05}

In [9]:
def run_season(start, end):
    cube = connection.datacube_from_process(
        "rainfall_total_by_zone",
        spatial_extent=kenya_bbox,
        temporal_extent=[start, end],
        geometries=selected,
    )
    records = json.loads(cube.save_result(format="JSON").download())
    df = pd.DataFrame(records)
    return df[df["variable"] == "mean"].set_index("county")["value"]

comparison = pd.DataFrame({
    "OND 2022 (mm)": run_season("2022-10-01", "2023-01-01"),
    "OND 2023 (mm)": run_season("2023-10-01", "2024-01-01"),
}).round(0)
print(comparison)

         OND 2022 (mm)  OND 2023 (mm)
county                               
Turkana           37.0          157.0
Wajir             74.0          486.0
Nakuru           461.0          316.0
Kitui            287.0          690.0


The same saved graph produced both columns; only the parameter values changed.

## 6. Build on a UDP

A UDP can return a raster cube instead of a final result. You can then use it like any other cube: do arithmetic on it, pass it to other processes, or call it from inside another UDP. Before running a graph, the server replaces each UDP call with the UDP's own nodes.

Here we save a smaller UDP that returns the seasonal rainfall total as a raster. We call it twice and subtract the results to map the change from 2022 to 2023.

In [10]:
season_total = connection.load_collection(
    CHIRPS,
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
).reduce_dimension(dimension="time", reducer="sum")

connection.save_user_defined_process(
    user_defined_process_id="seasonal_rain_total",
    process_graph=season_total,
    parameters=[spatial_extent, temporal_extent],
    summary="CHIRPS v3 rainfall total over a period (raster)",
)

ond_2022 = connection.datacube_from_process(
    "seasonal_rain_total", spatial_extent=kenya_bbox, temporal_extent=["2022-10-01", "2023-01-01"])
ond_2023 = connection.datacube_from_process(
    "seasonal_rain_total", spatial_extent=kenya_bbox, temporal_extent=["2023-10-01", "2024-01-01"])

change = ond_2023 - ond_2022
change_by_county = change.process("zonal_stats", data_cube=change, geometries=selected, stats=["mean"])

records = json.loads(change_by_county.save_result(format="JSON").download())
df = pd.DataFrame(records)
print(df[df["variable"] == "mean"].set_index("county")["value"].round(0).rename("Change 2023 vs 2022 (mm)"))

county
Turkana    120.0
Wajir      412.0
Nakuru    -146.0
Kitui      403.0
Name: Change 2023 vs 2022 (mm), dtype: float64


## 7. Run a Process Graph from JSON

Graphs built in the Web Editor, shared by colleagues, or saved in files are plain JSON. `datacube_from_json` loads such a graph so you can run it from Python. The graph below uses the same structure as section 2, but with a different area: central Kenya around Mount Kenya.

In [11]:
graph_json = """
{
  "process_graph": {
    "load": {
      "process_id": "load_collection",
      "arguments": {
        "id": "fao-gismgr:CHIRPS-V3:raster:mapsets:EWX2-DATA-GM",
        "spatial_extent": {"west": 36.8, "east": 37.8, "south": -0.6, "north": 0.4},
        "temporal_extent": ["2023-10-01", "2024-01-01"]
      }
    },
    "total": {
      "process_id": "reduce_dimension",
      "arguments": {
        "data": {"from_node": "load"},
        "dimension": "time",
        "reducer": {"process_graph": {
          "sum": {"process_id": "sum", "arguments": {"data": {"from_parameter": "data"}}, "result": true}
        }}
      }
    },
    "save": {
      "process_id": "save_result",
      "arguments": {"data": {"from_node": "total"}, "format": "GTiff"},
      "result": true
    }
  }
}
"""

cube = connection.datacube_from_json(graph_json)
cube.download("mount_kenya_ond2023.tif")
print("Downloaded mount_kenya_ond2023.tif")

Downloaded mount_kenya_ond2023.tif


## 8. Use Your UDPs in the Web Editor

1. Open the [FAO OpenEO Web Editor](https://data.fao.org/editor/?server=data.fao.org%2Fopeneo) and sign in with the same account.
2. Click the **User-Defined Processes** tab in the bottom panel. Your saved processes (`rainfall_total_by_zone`, `seasonal_rain_total`) are listed there, with buttons to view details, edit and delete them. The info view shows the summary, description and each parameter with its type, exactly as declared in section 3.
3. They also appear in the **Processes** list in the left panel, next to the predefined processes. Search for the ID and drag the process into the **Visual Model**.
4. Open the process node to fill in its parameters. The inputs come from the schemas declared in section 3: **Spatial Extent** is a bounding box you draw on a map, **Temporal Extent** has start and end date pickers, and **Geometries** is a map with drawing tools for polygons (or you can supply GeoJSON). Required parameters are marked with a red asterisk, and each shows the description you gave it. Click **Submit** to save the values.
5. Connect the process to `save_result`, then use **Run directly** (synchronous) or **Create Batch Job** (longer runs) below the model.
6. To go the other way, build a graph in the Editor, open the **Code** tab, copy the JSON and use it with `datacube_from_json` as in section 7. To bring JSON into the Editor, paste it into the Visual Model (the paste button, or Ctrl/Cmd+V), drop a `.json` file onto it, or use the import button.

## 9. Clean Up (optional)

The two UDPs stay in your account, so you can look at them in the Web Editor as described above. When you no longer need them, run the cell below to delete them. It's commented out so that running the whole notebook doesn't remove them.

In [12]:
# Uncomment to delete the UDPs created in this tutorial
# for udp_id in ["rainfall_total_by_zone", "seasonal_rain_total"]:
#     connection.user_defined_process(udp_id).delete()

print("Your UDPs:", [p["id"] for p in connection.list_user_defined_processes()])

Your UDPs: ['seasonal_rain_total', 'rainfall_total_by_zone']


## Summary

In this tutorial, you learned how to:

1. **Read a process graph** as JSON with `to_json()`
2. **Declare parameters** with `Parameter.spatial_extent`, `temporal_interval` and `geojson`
3. **Save, list and inspect** user-defined processes
4. **Call a UDP** with different inputs using `datacube_from_process`
5. **Build on a UDP** by doing arithmetic on its output
6. **Run JSON process graphs** from the Web Editor or files with `datacube_from_json`

### Things to Remember

- Pass a value for every UDP parameter; defaults aren't applied.
- UDP IDs can contain only letters, digits and underscores.
- A UDP must have exactly one result node.
- UDPs are private to your account.

### Related Tutorials

- **[Getting Started](01_getting_started.ipynb)**: connecting, sync and batch execution
- **[Drought Monitoring](07_drought_monitoring.ipynb)**: a full analysis you could save as a UDP